In [ ]:
# Ultralisk-v2.3 | ShadowBytes | ApuCast | Kaggle T4 x2
import os, time, json, copy, hashlib, threading, platform
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display, FileLink
SMOKE=os.environ.get('ULTRALISK_SMOKE')=='1'  # prueba tecnica, NO subir
OUT=Path(os.environ.get('ULTRALISK_OUT','/kaggle/working/Ultralisk-v2.3')); OUT.mkdir(parents=True,exist_ok=True)
LOG=OUT/'training.log'; LOG.write_text(''); LOCK=threading.Lock(); START=time.perf_counter()
BATCH=32; EPOCHS_REF=25; EPOCHS_NEW=36
SEEDS={'CNN':(0,1,2),'ResNet':(0,1,2),'Mutalisk':(42,123),'Viper':(42,123)}
CUTS=[('2024050900','2024051600','2024060100'),('2024052500','2024060100','2024061600'),('2024060900','2024061600','2024070100')]
VARS=['RH','T','U','V']; NORM=np.array([25.360,2.0652,2.6776,2.0876])
# Valores historicos de versions/v3/backtest.json (solo referencia, no etiquetas).
HISTORICAL={1:.42292221906027727,2:.4136267261457105,3:.42723285641743414}
MIN_DEV_GAIN=.003; MIN_AUDIT_GAIN=.002
def log(msg):
    line=f'[{time.perf_counter()-START:8.1f}s] {msg}'
    with LOCK:
        print(line,flush=True)
        with LOG.open('a') as f: f.write(line+'\n')
if not SMOKE and (torch.cuda.device_count()!=2 or any('T4' not in torch.cuda.get_device_name(i) for i in range(2))): raise RuntimeError('Selecciona Accelerator > GPU T4 x2')
DEVICES=[torch.device('cpu')] if SMOKE else [torch.device('cuda:0'),torch.device('cuda:1')]
torch.set_num_threads(2); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
cpu_name=platform.processor()
if Path('/proc/cpuinfo').is_file():
    cpu_name=next((l.split(':',1)[1].strip() for l in Path('/proc/cpuinfo').read_text().splitlines() if l.startswith('model name')),cpu_name)
log(f'torch={torch.__version__} CPU={cpu_name} cores={os.cpu_count()} torch_threads={torch.get_num_threads()} smoke={SMOKE}')
for d in DEVICES:
    if d.type=='cuda': log(f'{d} {torch.cuda.get_device_name(d)} VRAM={torch.cuda.get_device_properties(d).total_memory/2**30:.2f} GiB')
# Datos oficiales
root=Path(os.environ.get('ULTRALISK_DATA','/kaggle/input')); required=('train.npz','test.npz','static.npz','sample_submission.csv')
folders=[p.parent for p in root.rglob('train.npz') if all((p.parent/k).is_file() for k in required)]
official=[p for p in folders if 'hackathon-ia-pucp-2026' in str(p)]
if len(official)==1: DATA=official[0]
elif len(folders)==1: DATA=folders[0]
else: raise RuntimeError(f'Dataset oficial no identificado: {folders}')
with np.load(DATA/'train.npz') as f: X,Y,TT=f['X'].astype('f4'),f['Y'].astype('f4'),f['times'].astype(str)
with np.load(DATA/'test.npz') as f: XT,TE=f['X'].astype('f4'),f['times'].astype(str)
with np.load(DATA/'static.npz') as f:
    DEM=f['dem'].astype('f4'); glat,glon,wlat,wlon=[f[k] for k in ('gfs_lats','gfs_lons','wrf_lats','wrf_lons')]
TEMPLATE=pd.read_csv(DATA/'sample_submission.csv',dtype={'id':str})
assert X.shape==(2935,17,13,4) and Y.shape==(2935,70,35,4) and XT.shape==(792,17,13,4) and DEM.shape==(70,35)
assert TEMPLATE.shape==(646800,5) and TEMPLATE.columns.tolist()==['id',*VARS] and TEMPLATE.id.is_unique
assert all(np.isfinite(a).all() for a in (X,Y,XT,DEM)) and max(TT)<min(TE)
parts=TEMPLATE.id.str.split('_',expand=True); ids_t=parts[0].map({s:i for i,s in enumerate(TE)}); assert ids_t.notna().all()
ti=ids_t.to_numpy(dtype=int); ri=parts[1].astype(int).to_numpy(); ci=parts[2].astype(int).to_numpy()
assert (ri>=0).all() and (ri<70).all() and (ci>=0).all() and (ci<35).all()
assert len(np.unique(ti))==264 and np.all(np.bincount(ti)[np.unique(ti)]==2450)
HOURS=np.unique([int(s[-2:]) for s in parts[0].unique()]); H=np.array([int(s[-2:]) for s in TT]); HT=np.array([int(s[-2:]) for s in TE])
log(f'Data={DATA} train={X.shape} test={XT.shape} horas={HOURS.tolist()}')
def windows(x,times):
    dates=pd.to_datetime(times,format='%Y%m%d%H'); lookup={t:i for i,t in enumerate(dates)}
    return x[np.array([[lookup.get(t-pd.Timedelta(hours=k),i) for k in (2,1,0)] for i,t in enumerate(dates)])]
W,WT=windows(X,TT),windows(XT,TE)
lat,lon=np.meshgrid(wlat,wlon,indexing='ij'); grid=np.stack([(lon-glon[0])/(glon[-1]-glon[0])*2-1,(lat-glat[0])/(glat[-1]-glat[0])*2-1],-1)[None].astype('f4')
def z(a): return ((a-a.mean())/max(float(a.std()),1e-6)).astype('f4')
def smooth(a,k):
    p=k//2; return F.avg_pool2d(F.pad(torch.from_numpy(a[None,None]),(p,p,p,p),mode='replicate'),k,stride=1)[0,0].numpy()
dy,dx=np.gradient(DEM)
geo=np.stack([z(DEM),z(dx),z(dy),z(np.hypot(dx,dy)),z(DEM-smooth(DEM,3)),z(DEM-smooth(DEM,7)),z(DEM-smooth(DEM,15)),z(lat),z(lon),z(np.gradient(dy,axis=0)+np.gradient(dx,axis=1))])[None].astype('f4')
def stats(ix):
    yy,xx,hh=Y[ix],X[ix],H[ix]
    def climate(a): return np.stack([a[hh==h].mean(0) if np.any(hh==h) else a.mean(0) for h in range(24)])
    cl=climate(yy)
    return dict(mu=xx.mean((0,1,2)),sd=np.maximum(xx.std((0,1,2)),1e-6),clim=cl,gfs_clim=climate(xx),scale=np.maximum((yy-cl[hh]).std((0,1,2)),1e-6))
def score(p,y):
    rm=np.sqrt(np.mean(np.square(p.astype('f8')-y),axis=(0,1,2))); return float(1-(rm/NORM).mean()),rm
def clip(p):
    p=p.astype('f4',copy=False); p[...,0]=p[...,0].clip(0,100); assert np.isfinite(p).all(); return p
def tensor(a,d): return torch.as_tensor(a,dtype=torch.float32,device=d)
def sync(d):
    if d.type=='cuda': torch.cuda.synchronize(d)
# Arquitecturas
class Residual(nn.Module):
    def __init__(self,a,b,dilation=1):
        super().__init__(); self.c1=nn.Conv2d(a,b,3,padding=dilation,dilation=dilation); self.c2=nn.Conv2d(b,b,3,padding=1); self.skip=nn.Conv2d(a,b,1) if a!=b else nn.Identity()
    def forward(self,x): return F.gelu(self.c2(F.gelu(self.c1(x)))+self.skip(x))
def inputs(x,h):
    n=len(x); a=h*(2*np.pi/24); clock=torch.stack([a.sin(),a.cos()],1)[:,:,None,None].expand(-1,-1,17,13)
    return x.permute(0,1,4,2,3).reshape(n,12,17,13),clock
class Reference(nn.Module):
    def __init__(self,kind):
        super().__init__(); w=64
        self.enc=nn.Sequential(nn.Conv2d(14,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU())
        self.static=nn.Parameter(torch.zeros(1,8,70,35))
        self.dec=nn.Sequential(nn.Conv2d(73,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU(),nn.Conv2d(w,4,1))
        if kind=='ResNet': self.enc=Residual(14,w); self.dec=nn.Sequential(Residual(73,w),nn.Conv2d(w,4,1))
        self.register_buffer('grid',torch.from_numpy(grid)); self.register_buffer('dem',torch.from_numpy(z(DEM)[None,None]))
    def forward(self,x,h):
        n=len(x); fields,clock=inputs(x,h); v=self.enc(torch.cat([fields,clock],1))
        with torch.autocast(device_type=x.device.type,enabled=False): v=F.grid_sample(v.float(),self.grid.expand(n,-1,-1,-1),align_corners=True)
        return self.dec(torch.cat([v,self.dem.expand(n,-1,-1,-1),self.static.expand(n,-1,-1,-1)],1)).permute(0,2,3,1)
class GeoWind(nn.Module):
    def __init__(self,kind,st):
        super().__init__(); self.kind=kind; w=48
        self.register_buffer('grid',torch.from_numpy(grid)); self.register_buffer('geo',torch.from_numpy(geo)); self.register_buffer('gcl',torch.from_numpy(((st['gfs_clim']-st['mu'])/st['sd']).astype('f4')))
        self.enc=nn.Sequential(Residual(22,w),Residual(w,w,2),Residual(w,w)); self.map=nn.Parameter(torch.zeros(1,8,70,35)); self.fuse=Residual(w+26,w)
        if kind=='Mutalisk': self.down=nn.Sequential(Residual(w,64),Residual(64,64,2)); self.up=Residual(w+64,w)
        else: self.wind=nn.Sequential(Residual(w,w,2),Residual(w,w,3))
        self.thermal=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1)); self.velocity=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1))
    def interp(self,x):
        with torch.autocast(device_type=x.device.type,enabled=False): return F.grid_sample(x.float(),self.grid.expand(len(x),-1,-1,-1),align_corners=True,padding_mode='border')
    def forward(self,x,h):
        n=len(x); fields,clock=inputs(x,h); current=x[:,-1]; anomaly=current-self.gcl[h.long()]; trend=current-x[:,0]
        av=anomaly.permute(0,3,1,2); cv=current.permute(0,3,1,2); v=self.enc(torch.cat([fields,av,trend.permute(0,3,1,2),clock],1))
        fine=self.fuse(torch.cat([self.interp(v),self.interp(torch.cat([cv,av],1)),self.geo.expand(n,-1,-1,-1),self.map.expand(n,-1,-1,-1)],1))
        if self.kind=='Mutalisk':
            low=self.down(F.avg_pool2d(fine,2)); fine=self.up(torch.cat([fine,F.interpolate(low,size=fine.shape[-2:],mode='bilinear',align_corners=False)],1)); wind=fine
        else: wind=self.wind(fine)
        return torch.cat([self.thermal(fine),self.velocity(wind)],1).permute(0,2,3,1)
def make(kind,st): return Reference(kind) if kind in ('CNN','ResNet') else GeoWind(kind,st)
# Benchmark de inferencia, mismo modelo y lote para todos los dispositivos.
def benchmark():
    st=stats(np.arange(128)); torch.manual_seed(0); base=Reference('ResNet').eval(); results=[]
    x=(W[:8]-st['mu'])/st['sd']; h=H[:8]
    for d,amp in [(torch.device('cpu'),False)]+[(d,a) for d in DEVICES if d.type=='cuda' for a in (False,True)]:
        m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)
        with torch.inference_mode():
            for _ in range(3):
                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)
            sync(d); t=time.perf_counter()
            for _ in range(8):
                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)
            sync(d); sec=time.perf_counter()-t
        r=dict(device=str(d),amp=amp,batch=8,ms_per_batch=sec/8*1000,samples_per_second=64/sec); results.append(r); log(f'BENCHMARK {r}')
    if len(DEVICES)==2:
        barrier=threading.Barrier(2)
        def both(d):
            m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)
            with torch.inference_mode(), torch.autocast(device_type='cuda',dtype=torch.float16):
                for _ in range(3): m(xb,hb)
                sync(d); barrier.wait(timeout=60); t=time.perf_counter()
                for _ in range(8): m(xb,hb)
                sync(d); return time.perf_counter()-t
        with ThreadPoolExecutor(max_workers=2) as pool:
            sec=max(pool.map(both,DEVICES))
        r=dict(device='T4x2_parallel',amp=True,batch=16,ms_per_batch=sec/8*1000,samples_per_second=128/sec)
        results.append(r); log(f'BENCHMARK {r}')
    return results
BENCH=benchmark()
@torch.inference_mode()
def infer(model,x,h,d,amp):
    model.eval(); out=[]
    for i in range(0,len(x),BATCH):
        with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[i:i+BATCH],h[i:i+BATCH])
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)
def fit_neural(job,d,st,tr,we,he,tag):
    kind,seed,model=job; ref=kind in ('CNN','ResNet'); epochs=1 if SMOKE else (EPOCHS_REF if ref else EPOCHS_NEW)
    amp=not ref and d.type=='cuda'; lr=.002 if ref else .0012; rng=np.random.RandomState(seed)
    model=model.to(d,memory_format=torch.channels_last); ema=None if ref else copy.deepcopy(model).eval().requires_grad_(False)
    x=tensor((W[tr]-st['mu'])/st['sd'],d); h=tensor(H[tr],d); y=tensor((Y[tr]-st['clim'][H[tr]])/st['scale'],d)
    xe=tensor((we-st['mu'])/st['sd'],d); he_gpu=tensor(he,d)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4 if ref else 5e-4)
    steps=(len(tr)+BATCH-1)//BATCH; sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,total_steps=steps*epochs)
    scaler=torch.amp.GradScaler('cuda',enabled=amp); ratio=tensor(st['scale']/NORM,d)
    order=np.arange(len(tr)); started=time.perf_counter(); hist=[]
    if d.type=='cuda': torch.cuda.reset_peak_memory_stats(d)
    log(f'{tag} {kind}:{seed} {d} START params={sum(p.numel() for p in model.parameters()):,} epochs={epochs} amp={amp}')
    for ep in range(1,epochs+1):
        model.train(); rng.shuffle(order); accum=torch.zeros((),device=d); t0=time.perf_counter()
        for k in range(0,len(order),BATCH):
            ix=torch.as_tensor(order[k:k+BATCH],device=d,dtype=torch.long); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[ix],h[ix])
            mse=(p.float()-y[ix]).square(); loss=mse.mean() if ref else .5*mse.mean()+.5*((mse.mean((0,1,2))+1e-8).sqrt()*ratio).mean()
            scaler.scale(loss).backward()
            if not ref: scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)
            old=scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale()>=old:
                sched.step()
                if ema is not None:
                    with torch.no_grad():
                        for a,b in zip(ema.parameters(),model.parameters()): a.lerp_(b,.015)
            accum+=loss.detach()*len(ix)
        sync(d); sec=time.perf_counter()-t0; value=float(accum.item()/len(tr)); assert np.isfinite(value)
        mem=torch.cuda.max_memory_allocated(d)/2**30 if d.type=='cuda' else None
        hist.append(dict(epoch=ep,loss=value,seconds=sec,samples_per_second=len(tr)/sec,peak_gib=mem,lr=opt.param_groups[0]['lr']))
        log(f'{tag} {kind}:{seed} {d} ep={ep:02d}/{epochs} loss={value:.5f} lr={opt.param_groups[0]["lr"]:.6f} {sec:.2f}s {len(tr)/sec:.1f} samples/s VRAM={mem}')
    chosen=model if ref else ema; p=clip(infer(chosen,xe,he_gpu,d,amp)*st['scale']+st['clim'][he]); elapsed=time.perf_counter()-started
    pd.DataFrame(hist).to_csv(OUT/f'{tag}_{kind}_{seed}_history.csv',index=False)
    if tag=='final': torch.save({'kind':kind,'seed':seed,'epochs':epochs,'state_dict':{k:v.detach().cpu() for k,v in chosen.state_dict().items()}},OUT/f'{kind}_{seed}.pt')
    log(f'{tag} {kind}:{seed} DONE {elapsed:.1f}s'); model.to('cpu')
    return kind,p,dict(kind=kind,seed=seed,device=str(d),seconds=elapsed,epochs=epochs,peak_gib=mem)
# Kernel Ridge no lineal: anomalias del GFS, tendencia y promedio temporal.
def kernel_features(w,h,st):
    a=(w-st['gfs_clim'][(h[:,None]-np.array([2,1,0]))%24])/st['sd']; a=a.reshape(len(w),3,-1)
    return np.concatenate([a[:,2],a.mean(1),.5*(a[:,2]-a[:,0])],axis=1).astype('f4')
def kernel(a,b):
    dot=(a@b.T)/a.shape[1]; dist=(a.square().mean(1)[:,None]+b.square().mean(1)[None,:]-2*dot).clamp_min(0)
    return .5*dot+.25*dot.square()+.25*torch.exp(-dist/2)
def fit_kernel(d,st,tr,we,he,tag):
    t=time.perf_counter(); a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)
    mu=a.mean(0); sd=np.maximum(a.std(0),.15); a=(a-mu)/sd; b=(b-mu)/sd
    ag,bg=tensor(a,d),tensor(b,d); K=kernel(ag,ag); K=(K+K.T)*.5; K.diagonal().add_(.05)
    L=torch.linalg.cholesky(K); yy=tensor(((Y[tr]-st['clim'][H[tr]])/st['scale']).reshape(len(tr),-1),d)
    dual=torch.cholesky_solve(yy,L); p=(kernel(bg,ag)@dual).cpu().numpy().reshape(len(we),70,35,4)
    if tag=='final': np.savez_compressed(OUT/'kernel_model.npz',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy())
    log(f'{tag} Kernel DONE {time.perf_counter()-t:.2f}s alpha=.05'); return clip(p*st['scale']+st['clim'][he])
ALL_RUNS=[]
def run_stage(tr,we,he,tag,families):
    st=stats(tr); jobs=[]
    for kind in families:
        if kind=='Kernel': continue
        for seed in (SEEDS[kind][:1] if SMOKE else SEEDS[kind]):
            torch.manual_seed(seed); jobs.append((kind,seed,make(kind,st)))
    def worker(group,d): return [fit_neural(j,d,st,tr,we,he,tag) for j in group]
    results=[]
    if len(DEVICES)==2:
        with ThreadPoolExecutor(max_workers=2) as pool:
            futures=[pool.submit(worker,jobs[i::2],d) for i,d in enumerate(DEVICES)]
            for f in futures: results.extend(f.result())
    else: results=worker(jobs,DEVICES[0])
    ALL_RUNS.extend([dict(tag=tag,**r) for _,_,r in results]); results.sort(key=lambda x:x[2]['seed'])
    pred={k:np.mean([p for kind,p,_ in results if kind==k],axis=0,dtype=np.float32) for k in families if k!='Kernel'}
    if 'Kernel' in families: pred['Kernel']=fit_kernel(DEVICES[0],st,tr,we,he,tag)
    if tag=='final': np.savez_compressed(OUT/'preprocessing.npz',**st,grid=grid,geo=geo,normalizers=NORM)
    return pred
# Calibracion por variable, separada de evaluacion.
CANDIDATES={'Reference':{},'Mutalisk':{'Mutalisk':1.},'Viper':{'Viper':1.},'Neural':{'Mutalisk':.5,'Viper':.5},'Kernel':{'Kernel':1.},'Hybrid':{'Mutalisk':.375,'Viper':.375,'Kernel':.25}}
def fit_alpha(a,b,y,prior=.5,shrink=.2,cap=1.):
    d=b.astype('f8')-a; r=y.astype('f8')-a; den=(d*d).mean((0,1,2)); num=(r*d).mean((0,1,2))
    raw=np.divide(num,den,out=np.full(4,prior),where=den>1e-12); return np.clip((1-shrink)*raw+shrink*prior,0,cap).astype('f4')
def mix(a,b,w): return clip(a*(1-w)+b*w)
def candidate(p,name): return sum(p[k]*np.float32(w) for k,w in CANDIDATES[name].items())
def calibrate(p,y,name):
    rw=fit_alpha(p['CNN'],p['ResNet'],y); ref=mix(p['CNN'],p['ResNet'],rw)
    alpha=np.zeros(4,dtype='f4') if name=='Reference' else fit_alpha(ref,candidate(p,name),y,prior=.25,shrink=.35,cap=.65)
    return rw,alpha
def combine(p,name,rw,alpha):
    ref=mix(p['CNN'],p['ResNet'],rw); return ref if name=='Reference' else mix(ref,candidate(p,name),alpha)
CAL=[]; ROWS=[]; families=['CNN','ResNet','Mutalisk','Viper','Kernel']
for fold,(cal,start,end) in enumerate(CUTS,1):
    tr=np.flatnonzero(TT<cal); ca=np.flatnonzero((TT>=cal)&(TT<start)&np.isin(H,HOURS)); ev=np.flatnonzero((TT>=start)&(TT<end)&np.isin(H,HOURS))
    if SMOKE: tr,ca,ev=tr[:48],ca[:4],ev[:4]
    assert len(tr) and len(ca) and len(ev) and max(TT[tr])<min(TT[ca])<min(TT[ev])
    log(f'FOLD {fold} train={len(tr)} calibration={len(ca)} evaluation={len(ev)} {cal}/{start}/{end}')
    stage_families=families if fold<3 else ['CNN','ResNet']+list(CANDIDATES[SELECTED])
    ix=np.r_[ca,ev]; p=run_stage(tr,W[ix],H[ix],f'fold{fold}',stage_families)
    pc={k:v[:len(ca)].copy() for k,v in p.items()}; pe={k:v[len(ca):] for k,v in p.items()}; CAL.append((pc,Y[ca]))
    for name in (CANDIDATES if fold<3 else dict.fromkeys(['Reference',SELECTED])):
        rw,alpha=calibrate(pc,Y[ca],name); pred=combine(pe,name,rw,alpha); s,r=score(pred,Y[ev])
        ROWS.append(dict(fold=fold,model=name,skill=s,**dict(zip(['RMSE_'+v for v in VARS],r.tolist()))))
        log(f'FOLD {fold} {name:10s} SKILL={s:.6f} RMSE={np.round(r,4).tolist()} new_weights={alpha.round(3).tolist()}')
    del p,pe
    if fold==2:
        tab=pd.DataFrame(ROWS).pivot(index='model',columns='fold',values='skill'); gains=tab.sub(tab.loc['Reference'],axis=1)
        historical_gain=tab.sub(pd.Series({1:HISTORICAL[1],2:HISTORICAL[2]}),axis=1)
        eligible=[n for n in CANDIDATES if n!='Reference' and gains.loc[n].mean()>=MIN_DEV_GAIN and gains.loc[n].min()>=0 and (SMOKE or (historical_gain.loc[n].mean()>=MIN_DEV_GAIN and historical_gain.loc[n].min()>=0))]
        SELECTED=max(eligible,key=lambda n:gains.loc[n].mean()) if eligible else 'Reference'
        log(f'FROZEN BEFORE AUDIT: {SELECTED} dev gains={gains.loc[SELECTED].to_dict()}')
TABLE=pd.DataFrame(ROWS); TABLE.to_csv(OUT/'validation_metrics.csv',index=False)
PIVOT=TABLE.pivot(index='model',columns='fold',values='skill'); print(PIVOT.round(6).to_string(),flush=True)
GAIN=float(PIVOT.loc[SELECTED,3]-PIVOT.loc['Reference',3]); HIST_GAIN=float(PIVOT.loc[SELECTED,3]-HISTORICAL[3]); PASS=SELECTED!='Reference' and GAIN>=MIN_AUDIT_GAIN and (SMOKE or HIST_GAIN>=MIN_AUDIT_GAIN)
FINAL=SELECTED if PASS else 'Reference'; log(f'AUDIT selected={SELECTED} gain={GAIN:+.6f} historical_gain={HIST_GAIN:+.6f} gate={PASS} final={FINAL}')
cal_p={k:np.concatenate([p[k] for p,y in CAL]) for k in ['CNN','ResNet']+list(CANDIDATES[FINAL])}; cal_y=np.concatenate([y for p,y in CAL]); RW,ALPHA=calibrate(cal_p,cal_y,FINAL)
log(f'FINAL weights ResNet={RW.tolist()} new={ALPHA.tolist()}')
# Reentrenamiento completo y CSV
needed=['CNN','ResNet']+list(CANDIDATES[FINAL]); tr=np.argsort(TT); eval_t=np.unique(ti)
if SMOKE: tr=tr[:48]
p=run_stage(tr,WT[eval_t],HT[eval_t],'final',needed); forecast=combine(p,FINAL,RW,ALPHA)
positions=np.full(len(TE),-1,dtype=int); positions[eval_t]=np.arange(len(eval_t))
values=forecast[positions[ti],ri,ci]; sub=pd.DataFrame(values.round(4),columns=VARS); sub.insert(0,'id',TEMPLATE.id)
name='submission_SMOKE_NO_SUBIR.csv' if SMOKE else ('submission_ultralisk_v2_3.csv' if PASS else 'submission_reference_NO_SUBIR.csv')
dest=OUT/name; tmp=dest.with_suffix('.tmp'); sub.to_csv(tmp,index=False)
check=pd.read_csv(tmp,dtype={'id':str}); assert check.shape==TEMPLATE.shape and check.columns.tolist()==TEMPLATE.columns.tolist() and check.id.equals(TEMPLATE.id) and check.id.is_unique
assert np.isfinite(check[VARS].to_numpy()).all() and check.RH.between(0,100).all(); tmp.replace(dest)
report=dict(model='Ultralisk-v2.3',smoke=SMOKE,selected_before_audit=SELECTED,final=FINAL,audit_gain=GAIN,historical_audit_gain=HIST_GAIN,historical_backtest=HISTORICAL,cpu=cpu_name,torch_threads=torch.get_num_threads(),recommend_submit=bool(PASS and not SMOKE),reference_public_score=.46485,normalizers=NORM.tolist(),reference_resnet_weights=RW.tolist(),new_weights=ALPHA.tolist(),new_family_mix=CANDIDATES[FINAL],epochs_ref=EPOCHS_REF,epochs_new=EPOCHS_NEW,seeds=SEEDS,benchmark=BENCH,runs=ALL_RUNS,validation=ROWS,sha256=hashlib.sha256(dest.read_bytes()).hexdigest(),rows=len(check),seconds=time.perf_counter()-START,torch=torch.__version__)
(OUT/'report.json').write_text(json.dumps(report,indent=2))
log(f'CSV={dest} SHA256={report["sha256"]} rows={len(check)}')
log('RECOMMEND_SUBMIT=YES' if report['recommend_submit'] else 'RECOMMEND_SUBMIT=NO: conservar 0.46485')
for f in (dest,OUT/'report.json',OUT/'validation_metrics.csv',LOG): display(FileLink(os.path.relpath(f)))
